# Simple Health Care Chatbot for Google Colab

This notebook builds a beginner-friendly health care chatbot in plain Python.

**What it can do**
- Answer common health questions with short, safety-focused guidance.
- Recognize urgent warning signs and tell the user to seek emergency care.
- Use a small built-in health knowledge base for fast answers.
- Search trusted public web summaries when the question is not in the built-in data.
- Remind users that it is not a doctor and cannot diagnose disease.

> **Important:** This chatbot is for education only. It does not replace a licensed doctor, nurse, pharmacist, or emergency service. If symptoms are severe, sudden, worsening, or life-threatening, call your local emergency number immediately.


In [ ]:
import json
import re
import textwrap
import urllib.parse
import urllib.request

print('✅ Health care chatbot is ready to load.')


## Chatbot Code
Run the cell below. The code stays simple: it uses safety rules, a small medical knowledge base, and public web summaries for broader health questions.


In [ ]:
DISCLAIMER = (
    "I can share general health education, but I cannot diagnose or replace a clinician. "
    "For personal medical advice, contact a doctor, nurse, or pharmacist."
)

USER_AGENT = "SimpleHealthChatbotColab/2.0 (educational notebook)"

EMERGENCY_KEYWORDS = {
    "chest pain": "Chest pain can be serious, especially with shortness of breath, sweating, nausea, or pain spreading to the arm/jaw/back.",
    "can't breathe": "Trouble breathing can become life-threatening.",
    "cannot breathe": "Trouble breathing can become life-threatening.",
    "shortness of breath": "Shortness of breath can be serious if severe, sudden, or new.",
    "stroke": "Stroke symptoms need emergency care quickly.",
    "face drooping": "Face drooping can be a stroke warning sign.",
    "weakness on one side": "One-sided weakness can be a stroke warning sign.",
    "suicidal": "Feeling suicidal is an emergency and you deserve immediate support.",
    "overdose": "Possible overdose needs urgent medical help.",
    "severe bleeding": "Severe bleeding needs urgent medical attention.",
    "unconscious": "Unconsciousness is an emergency.",
    "seizure": "A first seizure, long seizure, or repeated seizures need urgent care.",
    "severe allergic reaction": "Severe allergic reactions can block breathing.",
    "anaphylaxis": "Anaphylaxis is a medical emergency.",
}

# Fast built-in answers for common questions. If a topic is not here, the bot searches public web summaries.
HEALTH_TOPICS = {
    "fever": {
        "answer": "A fever is a body temperature higher than normal, often caused by infection. Rest, drink fluids, and consider acetaminophen/paracetamol or ibuprofen if you can take them safely.",
        "care": "Seek medical care for fever in a baby under 3 months, fever lasting more than 3 days, very high fever, stiff neck, confusion, rash, dehydration, breathing trouble, or if you are immunocompromised."
    },
    "cough": {
        "answer": "A cough is commonly caused by a cold, flu, allergies, asthma, reflux, or other respiratory irritation. Warm fluids, honey for adults and children over 1 year, and avoiding smoke may help.",
        "care": "Get medical advice if cough lasts more than 3 weeks, produces blood, causes chest pain, comes with high fever, wheezing, shortness of breath, or affects a young child or frail adult."
    },
    "headache": {
        "answer": "Common headaches can be related to stress, dehydration, poor sleep, eyestrain, sinus problems, or migraine. Rest, water, a quiet dark room, and safe pain relief may help.",
        "care": "Seek urgent care for sudden worst headache, headache after head injury, fever with stiff neck, confusion, weakness, vision loss, pregnancy, or a new severe headache after age 50."
    },
    "burn": {
        "answer": "For a minor burn, cool the area under running cool water for about 20 minutes, remove tight items, cover with a clean non-stick dressing, and do not apply ice or butter.",
        "care": "Get urgent care for large, deep, chemical, electrical, face/hand/genital burns, burns in babies/older adults, or signs of infection."
    },
    "medicine": {
        "answer": "Take medicines exactly as prescribed or as the label says. Check dose, timing, allergies, interactions, pregnancy safety, and whether to take with food.",
        "care": "Ask a pharmacist or clinician before mixing medicines, giving medicine to a child, using medicine while pregnant, or if you have kidney/liver disease. For overdose, call emergency services or poison control."
    },
}

HEALTH_WORDS = {
    "ache", "allergy", "antibiotic", "blood", "body", "breathing", "cancer", "cold", "cough", "covid", "diabetes",
    "diagnosis", "diet", "disease", "doctor", "dose", "drug", "fever", "flu", "health", "heart", "hospital",
    "infection", "injury", "medicine", "medication", "mental", "nausea", "pain", "patient", "pill", "pregnancy",
    "rash", "sick", "symptom", "tablet", "treatment", "vaccine", "virus", "vomit", "wound"
}

GREETINGS = {"hi", "hello", "hey", "good morning", "good afternoon", "good evening"}
GOODBYES = {"bye", "exit", "quit", "goodbye"}


def clean_text(text):
    return re.sub(r"\s+", " ", text.lower().strip())


def fetch_json(url, timeout=10):
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def simplify_question(user_text):
    words_to_remove = ["what is", "what are", "tell me about", "explain", "symptoms of", "treatment for", "causes of", "medicine for", "how to treat"]
    topic = user_text
    for phrase in words_to_remove:
        topic = topic.replace(phrase, " ")
    topic = re.sub(r"[^a-z0-9 +\-]", " ", topic)
    topic = re.sub(r"\s+", " ", topic).strip()
    return topic or user_text


def is_health_related(user_text):
    if find_topic(user_text):
        return True
    return any(word in user_text for word in HEALTH_WORDS)


def find_topic(user_text):
    for topic in HEALTH_TOPICS:
        if topic in user_text:
            return topic
    return None


def check_emergency(user_text):
    for phrase, reason in EMERGENCY_KEYWORDS.items():
        if phrase in user_text:
            return phrase, reason
    return None, None


def answer_from_duckduckgo(user_text):
    query = simplify_question(user_text) + " health medicine"
    url = "https://api.duckduckgo.com/?" + urllib.parse.urlencode({
        "q": query,
        "format": "json",
        "no_html": 1,
        "skip_disambig": 1,
    })
    data = fetch_json(url)
    answer = data.get("AbstractText") or data.get("Answer")
    source = data.get("AbstractURL") or data.get("AbstractSource")
    if answer:
        return f"{answer} Source: {source}" if source else answer
    return None


def answer_from_wikipedia(user_text):
    topic = simplify_question(user_text)
    search_url = "https://en.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
        "action": "opensearch",
        "search": topic,
        "limit": 1,
        "namespace": 0,
        "format": "json",
    })
    search_data = fetch_json(search_url)
    if len(search_data) < 2 or not search_data[1]:
        return None

    title = search_data[1][0]
    summary_url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + urllib.parse.quote(title)
    summary = fetch_json(summary_url)
    extract = summary.get("extract")
    page_url = summary.get("content_urls", {}).get("desktop", {}).get("page")
    if extract:
        return f"{extract} Source: {page_url}" if page_url else extract
    return None


def web_health_answer(user_text):
    for answer_function in (answer_from_duckduckgo, answer_from_wikipedia):
        try:
            answer = answer_function(user_text)
            if answer:
                return answer
        except Exception:
            pass
    return None


def health_chatbot(message):
    text = clean_text(message)

    if not text:
        return "Please type a health question, symptom, medicine name, or condition. " + DISCLAIMER

    if text in GOODBYES:
        return "Take care. If you feel seriously unwell, seek medical help promptly."

    if text in GREETINGS:
        return "Hello! Ask me any health or medicine question. " + DISCLAIMER

    phrase, reason = check_emergency(text)
    if phrase:
        return (
            f"🚨 Possible emergency: {reason} "
            "Please call your local emergency number now or go to the nearest emergency department. "
            "If you are in the U.S. and this is a mental health crisis, call or text 988."
        )

    topic = find_topic(text)
    if topic:
        info = HEALTH_TOPICS[topic]
        return f"{info['answer']} {info['care']} {DISCLAIMER}"

    # If the question is not in the small built-in list, search public summaries.
    # This lets the bot answer many more health and medicine questions without a large hard-coded dataset.
    web_answer = web_health_answer(text)
    if web_answer:
        return f"{web_answer} {DISCLAIMER}"

    if not is_health_related(text):
        return "Please ask a question related to health, symptoms, diseases, medicines, first aid, or wellness. " + DISCLAIMER

    return (
        "I could not find a reliable short answer right now. Try asking with the condition or medicine name, such as "
        "'What is asthma?' or 'What are ibuprofen side effects?' If symptoms are serious or worsening, contact a clinician. "
        + DISCLAIMER
    )

print('✅ Chatbot loaded. It can answer built-in topics and search public summaries for many other health or medicine questions.')


## Try One Question
Run this cell and type one health question.


In [ ]:
question = input('Ask a health question: ')
print('\nBot:')
print(textwrap.fill(health_chatbot(question), width=100))


## Continuous Chat Mode
Run this cell to keep chatting. Type `bye`, `exit`, or `quit` to stop.


In [ ]:
print('🤖 Health Care Chatbot is running. Type bye, exit, or quit to stop.')

while True:
    user_message = input('You: ')
    bot_message = health_chatbot(user_message)
    print('Bot:', textwrap.fill(bot_message, width=100))
    if user_message.strip().lower() in GOODBYES:
        break


## Example Questions
Try these examples:

- `I have a fever`
- `What should I do for a cough?`
- `I have chest pain and shortness of breath`
- `How can I treat a small burn?`
- `What is asthma?`
- `What are ibuprofen side effects?`
- `Tell me about malaria treatment`
